In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import ml_tools as mlt
import sim_ranking as sr

In [ ]:
results_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/gnn/0817_1809")

db_ffp = Path("/Users/claudy/dev/work/data/sim_ranking/db/gm_db_neil.sqlite")

In [ ]:
run_config = sr.ml.gnn_gm.RunConfig.from_yaml(results_dir / "run_config.yaml")

db = sr.DB(db_ffp)
event_df = db.get_event_df()
gm_df = db.get_record_df()

In [ ]:
train_results = pd.read_parquet(results_dir / "train_results.parquet")
val_results = pd.read_parquet(results_dir / "val_results.parquet")

train_events = train_results["event_id"].unique()
train_results = pd.merge(train_results, gm_df, left_on=["event_id", "site_int"], right_on=["event_id", "site_id"], how="left", validate="one_to_one").drop(columns=["site_id"]).set_index(train_results.index)
train_results["num_obs"] = train_results["obs_sites"].apply(lambda x: len(x))

val_events = val_results["event_id"].unique()
val_results = pd.merge(val_results, gm_df, left_on=["event_id", "site_int"], right_on=["event_id", "site_id"], how="left", validate="one_to_one").drop(columns=["site_id"]).set_index(val_results.index)
val_results["num_obs"] = val_results["obs_sites"].apply(lambda x: len(x))

In [ ]:
# val_results.head()

In [ ]:
pred_im_keys = mlt.array_utils.numpy_str_join("_", run_config.ims, "pred")

## Validation Results


In [ ]:
#event_df.loc[val_events].sort_values("mag",ascending=False).head()

In [ ]:
# cur_event = "3468575"

In [ ]:
# val_results.loc[val_results.event_id == cur_event][["site_int", "obs_sites", "loss", "r_rup"]]

In [ ]:
def plot_pSA(id: str, results_df: pd.DataFrame, loglog: bool = False, ax: plt.Axes = None):
	cur_event = results_df.loc[id, "event_id"]
	cur_site_int = results_df.loc[id, "site_int"]
	
	fig = None
	if ax is None:
		fig, ax = plt.subplots(figsize=(8, 6))
	
	ax.plot(sr.constants.PERIODS, np.exp(results_df[pred_im_keys].loc[id].values), label="pred")
	ax.plot(sr.constants.PERIODS, np.exp(results_df[run_config.ims].loc[id].values), label="true")
	
	ax.set_title(f"Event: {cur_event}, Site of Interest: {cur_site_int}, Magnitude: {event_df.loc[cur_event].mag}, Rrup: {results_df.loc[id].r_rup:.1f}, Num Obs: {results_df.loc[id].num_obs}")
	ax.set_xlabel(f"Period (s)")
	ax.set_ylabel(f"pSA")
	ax.tick_params(axis="x", which="both", bottom=True)
	ax.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
	ax.set_xlim(0.01, 10)
	
	ax.set_xscale("log")
	if loglog:
		ax.set_yscale("log")
	
	ax.legend()
	
	plt.tight_layout()
	if fig is None:
		return ax
	
	return fig, ax

In [ ]:
# Well constrained site of interest
# plot_pSA("3468575_CCCC_XcjsV_X5Xa", val_results);

In [ ]:
# Poorly constrained site of interest
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
# plot_pSA("3468575_CSHS_isiMFYMtjT", val_results, ax=ax1);
# plot_pSA("3468575_CSHS_isiMFYMtjT", val_results, loglog=True, ax=ax2);

In [ ]:
n_plots = 25
ids = np.random.choice(val_results.index, n_plots, replace=False)
for id in ids:
	# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
	# plot_pSA(id, val_results, ax=ax1)
	# plot_pSA(id, val_results, loglog=True, ax=ax2)
	fig, ax = plot_pSA(id, val_results, loglog=True)
	plt.show()
	plt.close(fig)

## Train Results


In [ ]:
#event_df.loc[train_events].sort_values("mag",ascending=False).head()

In [ ]:
# cur_event = "3525264"
# train_results.loc[train_results.event_id == cur_event][["site_int", "obs_sites", "loss", "r_rup"]]

In [ ]:
## Well constrained
# plot_pSA("3525264_CACS_IuYhmeX6KJ", train_results);
# plot_pSA("3525264_CMHS_-YLKNV-5UI", train_results);

In [ ]:
# Poorly constrained
# plot_pSA("3525264_AMBC_OauqA5u0IQ", train_results);

In [ ]:
n_plots = 25
ids = np.random.choice(train_results.index, n_plots, replace=False)
for id in ids:
	# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
	# plot_pSA(id, train_results, ax=ax1)
	# plot_pSA(id, train_results, loglog=True, ax=ax2)
	fig, ax = plot_pSA(id, train_results, loglog=True)
	plt.show()
	plt.close(fig)

## Investigation into effect of number of observation and Rrup


In [ ]:
def plot_rrup_loss_scatter(results_df: pd.DataFrame):
	fig, ax = plt.subplots(figsize=(8, 6))
	
	cm = ax.scatter(results_df["r_rup"], results_df["loss"], c=results_df["num_obs"], cmap="viridis", s=10)
	
	ax.set_xlabel(f"Rrup")
	ax.set_ylabel(f"Loss")
	ax.tick_params(axis="x", which="both", bottom=True)
	ax.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")
	ax.set_xlim(1, 120)
	
	ax.set_yscale("log")
	ax.set_xscale("log")
	plt.colorbar(cm)
	
	plt.tight_layout()
	return fig, ax

### Validation


In [ ]:
plot_rrup_loss_scatter(val_results)
plt.show()

### Training


In [ ]:
plot_rrup_loss_scatter(train_results)
plt.show()